In [8]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA

import plotly.express as px


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

INPUT_FILE = "fighter_career_stats.csv"

K_CLUSTERS = 5
MIN_FIGHTS = 3
N_NEIGHBORS = 15
RANDOM_STATE = 42


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df_model = df[df["total_fights"] >= MIN_FIGHTS].copy()

print("Original fighters:", len(df))
print("Fighters after min-fight filter:", len(df_model))


# ------------------------------------------------------------
# 2. Select clustering features
# ------------------------------------------------------------

exclude_cols = [
    "fighter",            # ID column
    "total_fights",       # sample-size / reliability column
    "total_fight_time"    # career-volume column
]

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]

feature_cols = [
    col for col in feature_cols
    if pd.api.types.is_numeric_dtype(df_model[col])
]

print("Number of clustering features:", len(feature_cols))


# ------------------------------------------------------------
# 3. Preprocess features
# ------------------------------------------------------------

X_raw = df_model[feature_cols].replace([np.inf, -np.inf], np.nan)

preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

X_scaled = preprocess.fit_transform(X_raw)


# ------------------------------------------------------------
# 4. Spectral clustering
# ------------------------------------------------------------

spectral = SpectralClustering(
    n_clusters=K_CLUSTERS,
    affinity="nearest_neighbors",
    n_neighbors=N_NEIGHBORS,
    assign_labels="kmeans",
    random_state=RANDOM_STATE
)

df_model["style_cluster"] = spectral.fit_predict(X_scaled)


# ------------------------------------------------------------
# 5. Cluster counts and summary
# ------------------------------------------------------------

cluster_counts = df_model["style_cluster"].value_counts().sort_index()

cluster_summary = (
    df_model
    .groupby("style_cluster")[feature_cols]
    .mean()
    .round(3)
)

cluster_summary_table = cluster_summary.T
cluster_summary_table.index.name = "feature"

cluster_summary_table.columns = [
    f"Cluster {col}" for col in cluster_summary_table.columns
]

print("\nCluster counts:")
print(cluster_counts)

display(cluster_summary_table)






Original fighters: 2139
Fighters after min-fight filter: 1348
Number of clustering features: 53

Cluster counts:
style_cluster
0    233
1    466
2    199
3    255
4    195
Name: count, dtype: int64


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4
feature,,,,,
fight_quickness,0.587,0.719,0.627,0.712,0.585
win_DEC_pct,0.236,0.512,0.332,0.527,0.475
win_KO_TKO_pct,0.212,0.301,0.340,0.324,0.294
win_SUB_pct,0.410,0.138,0.211,0.102,0.218
loss_DEC_pct,0.423,0.528,0.444,0.495,0.239
loss_KO_TKO_pct,0.366,0.272,0.354,0.317,0.375
loss_SUB_pct,0.205,0.166,0.197,0.140,0.333
CTRL_pct,0.209,0.176,0.235,0.118,0.390
SIG_STR_acc,0.430,0.418,0.485,0.423,0.496


In [18]:
# ------------------------------------------------------------
# 5. Add readable cluster style labels
# ------------------------------------------------------------

cluster_style_names = {
    0: "Submission Grappler",
    1: "Defensive Decision Striker",
    2: "Clinch Pressure Striker",
    3: "High-Volume Distance Striker",
    4: "Control Wrestler"
}

df_model["style_label"] = df_model["style_cluster"].map(cluster_style_names)
# ------------------------------------------------------------
# 6. 3D PCA visualization with Plotly
# ------------------------------------------------------------

pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    "fighter": df_model["fighter"].values,
    "style_cluster": df_model["style_cluster"].values,
    "style_label": df_model["style_label"].values,
    "total_fights": df_model["total_fights"].values,
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "PC3": X_pca[:, 2]
})

fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="style_label",
    hover_name="fighter",
    hover_data={
        "style_cluster": True,
        "total_fights": True,
        "PC1": ":.3f",
        "PC2": ":.3f",
        "PC3": ":.3f"
    },
    title=f"3D PCA Visualization of Spectral Fighter Style Clusters, k={K_CLUSTERS}"
)

fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="style_label",
    hover_name="fighter",
    hover_data={
        "style_cluster": True,
        "total_fights": True,
        "PC1": ":.3f",
        "PC2": ":.3f",
        "PC3": ":.3f"
    },
    title=f"3D PCA Visualization of Spectral Fighter Style Clusters, k={K_CLUSTERS}",
    width=1000,
    height=800
)

fig.update_traces(
    marker=dict(
        size=3,
        opacity=1.0
    )
)

fig.show(
    config={
        "scrollZoom": True,
        "displayModeBar": True
    }
)